# FutureWork AI — Phase 3: Interactive Career Navigator
**IBM LinuxONE Datathon 2026 · Tech for Good**

This notebook turns the Phase 2 recommendation engine into an **interactive demonstration** with selection of a current occupation, personal skill self-assessment, explainable ranked occupations, skill charts and evaluation diagnostics.

**Interpretation:** O*NET values measure occupational *importance* (1–5), not an individual's proficiency. The similarity result is a **skills-profile similarity score, not a hiring probability**. Self-ratings are used only to flag learning priorities; they are not numerically subtracted from O*NET ratings. US O*NET descriptions must not be presented as verified Australian licensing rules.

## Before running
Place this notebook beside Phase 1 and Phase 2 so `processed/` contains:
- `occupation_skill_matrix.csv`
- `occupation_lookup.csv`
- `skill_dictionary.csv`
- `skills_long_clean.csv`

Run **Kernel → Restart & Run All**. `ipywidgets` is optional: if unavailable, a manual code-based demonstration remains possible. All processing runs inside JupyterLab on your IBM LinuxONE server.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, clear_output, Markdown
from sklearn.metrics.pairwise import cosine_similarity

ROOT = Path.cwd()
INPUT = ROOT / 'processed'
OUTPUT = ROOT / 'results'
OUTPUT.mkdir(exist_ok=True)
required = ['occupation_skill_matrix.csv', 'occupation_lookup.csv', 'skill_dictionary.csv', 'skills_long_clean.csv']
missing = [f for f in required if not (INPUT / f).is_file()]
if missing:
    raise FileNotFoundError('Run Phase 1 first. Missing processed files: ' + ', '.join(missing))
print('IBM LinuxONE workspace:', ROOT)

## 1. Load and validate occupational data

In [ ]:
raw = pd.read_csv(INPUT/'occupation_skill_matrix.csv', dtype={'occupation_code':str})
lookup = pd.read_csv(INPUT/'occupation_lookup.csv', dtype={'occupation_code':str})
dictionary = pd.read_csv(INPUT/'skill_dictionary.csv', dtype={'skill_id':str})
assert {'occupation_code'}.issubset(raw.columns)
assert {'occupation_code','occupation_title'}.issubset(lookup.columns)
assert {'skill_id','skill_name'}.issubset(dictionary.columns)
assert not raw['occupation_code'].duplicated().any(), 'Duplicate occupation IDs'
X = raw.set_index('occupation_code').apply(pd.to_numeric, errors='coerce')
lookup = lookup.drop_duplicates('occupation_code').set_index('occupation_code')
features = list(X.columns.intersection(dictionary.skill_id))
assert len(features) >= 2, 'No matching skill dimensions'
X = X.loc[X.index.intersection(lookup.index),features]
lookup = lookup.loc[X.index]
names = dictionary.drop_duplicates('skill_id').set_index('skill_id')['skill_name'].to_dict()
assert X.stack().between(1,5).all(), 'Expected 1-5 skill-importance values'
coverage = X.notna().mean(axis=1)
eligible = coverage[coverage >= .60].index
assert len(eligible) >= 6, 'Not enough eligible occupations'
filled = X.loc[eligible].fillna(X.loc[eligible].median().fillna(1))
vectors = np.maximum(filled.to_numpy(dtype=float)-1,0)
SIM = pd.DataFrame(cosine_similarity(vectors),index=eligible,columns=eligible)
print('Occupations:',len(X),'| Eligible:',len(eligible),'| Skills:',len(features))
print('Overall missing skill ratings:',f'{X.isna().mean().mean():.1%}')

## 2. Career recommender and transparent explanations

We compare occupation skill-importance vectors using cosine similarity. Missing values are median-imputed **for ranking only**; explanations use actual observed values.

In [ ]:
def recommend(source_code, top_n=5):
    source_code = str(source_code)
    if source_code not in SIM.index:
        raise ValueError('Select an eligible occupation')
    scores = SIM.loc[source_code].drop(source_code).sort_values(ascending=False).head(top_n)
    return pd.DataFrame({'occupation_code':scores.index,
                         'occupation_title':lookup.loc[scores.index,'occupation_title'].values,
                         'similarity':scores.values,
                         'data_coverage':coverage.loc[scores.index].values}).reset_index(drop=True)

def compare_skills(source_code,target_code,n=8):
    a,z = X.loc[str(source_code)],X.loc[str(target_code)]
    valid = a.notna() & z.notna()
    tab = pd.DataFrame({'skill_id':X.columns[valid], 'current_importance':a[valid].values,
                        'target_importance':z[valid].values})
    tab['skill'] = tab.skill_id.map(names).fillna(tab.skill_id)
    tab['delta_importance'] = tab.target_importance-tab.current_importance
    common = tab.assign(shared=tab[['current_importance','target_importance']].min(axis=1))
    common = common.sort_values(['shared','target_importance'],ascending=False).head(n)
    priorities = tab[tab.delta_importance>0].sort_values(['delta_importance','target_importance'],ascending=False).head(n)
    return common[['skill','current_importance','target_importance']], priorities[['skill','current_importance','target_importance','delta_importance']]

# Proof of basic ranking invariants
example_code = eligible[0]
example = recommend(example_code)
assert len(example)==5 and example_code not in set(example.occupation_code)
assert example.similarity.between(0,1.000001).all()
assert example.similarity.is_monotonic_decreasing
print('Recommendation engine validated.')

## 3. Visual report for one career transition

In [ ]:
def show_report(source_code,target_code=None,ratings=None,save=False):
    source_code = str(source_code)
    rec = recommend(source_code)
    if target_code not in set(rec.occupation_code):
        target_code = str(rec.iloc[0].occupation_code)
    current = lookup.loc[source_code,'occupation_title']
    target = lookup.loc[target_code,'occupation_title']
    display(Markdown(f'### {current} → career alternatives'))
    display(rec.style.format({'similarity':'{:.3f}','data_coverage':'{:.0%}'}))
    fig,ax = plt.subplots(figsize=(10,3.6))
    view = rec.iloc[::-1]
    ax.barh(view.occupation_title,view.similarity)
    ax.set_xlim(0,1);ax.set_xlabel('Skill-profile cosine similarity (NOT hiring probability)')
    ax.set_title('Top five related occupations')
    plt.tight_layout();display(fig);plt.close(fig)
    shared,gaps = compare_skills(source_code,target_code)
    display(Markdown(f'### Skill explanation: {current} → {target}'))
    display(Markdown('**Shared high-importance occupational skills**'))
    display(shared)
    display(Markdown('**Skills that are more important in the target occupation**'))
    display(gaps)
    if ratings:
        check = X.loc[target_code].dropna().sort_values(ascending=False).head(15)
        # User ratings are not mapped to occupational importance values mathematically.
        # We only flag self-reported development interest where the rating is low.
        flagged = pd.DataFrame([{'skill':names.get(s,s),'self_reported_confidence_1to5':int(ratings[s]),
                                 'target_importance_1to5':float(check[s])}
                                for s in check.index if s in ratings and ratings[s]<=2])
        display(Markdown('**Potential learning priorities (user self-rated confidence ≤2/5)**'))
        display(flagged if not flagged.empty else pd.DataFrame({'note':['None flagged among assessed skills']}))
    print('Limitations: Skills similarity does not measure eligibility, salary, job openings or likelihood of employment.')
    if save:
        rec.to_csv(OUTPUT/'interactive_recommendations.csv',index=False)
        shared.to_csv(OUTPUT/'interactive_shared_skills.csv',index=False)
        gaps.to_csv(OUTPUT/'interactive_skill_differences.csv',index=False)
        print('Saved demonstration results to:',OUTPUT)
    return rec

show_report(example_code,save=True)

## 4. Interactive demo: occupation and self-rated skills

Select an occupation, then click **Show recommendations**. Optionally self-assess a few skills. Your confidence ratings are **not** equal to O*NET job-importance ratings, and only flag learning areas to investigate. No personal information is saved automatically.

In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display
    ordered = sorted(((lookup.loc[c,'occupation_title'],c) for c in eligible),key=lambda x:x[0].lower())
    occupation_select = widgets.Combobox(
        placeholder='Type an occupation, then choose matching value below',
        options=[f'{title} | {code}' for title,code in ordered],
        value=f'{lookup.loc[example_code,"occupation_title"]} | {example_code}',
        description='Occupation:',ensure_option=True,layout=widgets.Layout(width='75%'),style={'description_width':'initial'})
    target_select = widgets.Dropdown(description='Explore match:',options=[('Highest ranked', '')],layout=widgets.Layout(width='75%'),style={'description_width':'initial'})
    skill_box = widgets.VBox()
    update_button = widgets.Button(description='Load top matches',button_style='info')
    show_button = widgets.Button(description='Show recommendations',button_style='success')
    out = widgets.Output()
    current_sliders = {}
    def selected_code():
        return occupation_select.value.rsplit(' | ',1)[-1]
    def load_options(*_):
        global current_sliders
        c = selected_code()
        if c not in SIM.index: return
        r = recommend(c)
        target_select.options=[(row.occupation_title,row.occupation_code) for row in r.itertuples()]
        # Select the source occupation's most important observed skills for self-report.
        ranked = X.loc[c].dropna().sort_values(ascending=False).head(6)
        current_sliders={s:widgets.IntSlider(value=3,min=1,max=5,description=names.get(s,s)[:32],
                        style={'description_width':'220px'},layout=widgets.Layout(width='550px')) for s in ranked.index}
        skill_box.children=tuple(current_sliders.values())
    def render(_):
        with out:
            clear_output(wait=True)
            c=selected_code()
            r={s:int(w.value) for s,w in current_sliders.items()}
            show_report(c,target_select.value,r,save=True)
    update_button.on_click(load_options)
    show_button.on_click(render)
    load_options()
    display(Markdown('**Self-rated confidence:** 1 = low, 5 = high; optional.'))
    display(widgets.VBox([occupation_select,update_button,target_select,skill_box,show_button,out]))
except ImportError:
    print('ipywidgets not available. Run the manual demonstration in the next section.')
except Exception as exc:
    print('Interactive widgets unavailable:',repr(exc))
    print('Use the manual demonstration below.')

## 5. Manual demonstration fallback

This always works, even when widget support is unavailable. Change the occupation search keyword, then rerun both cells.

In [ ]:
SEARCH = 'sales'  # Change to a keyword: nurse, teacher, programmer, etc.
matches = lookup.loc[eligible].loc[lambda d:d.occupation_title.str.contains(SEARCH,case=False,regex=False,na=False)]
display(matches.head(12).reset_index())
manual_source = matches.index[0] if len(matches) else example_code
print('Selected:',lookup.loc[manual_source,'occupation_title'])

In [ ]:
manual_results = show_report(manual_source,save=True)

## 6. Quality checks: invariants, coverage and sensitivity

These are **engineering validations**, not measured real-world recommendation accuracy. Precision@5 cannot be reported without an independent relevant-career reference set. In the demo, describe results as exploratory career similarities.

In [ ]:
# Test across multiple distinct eligible jobs (or fewer if dataset is tiny)
check_codes=list(eligible[:min(30,len(eligible))])
for c in check_codes:
    r=recommend(c)
    assert len(r)==5
    assert c not in set(r.occupation_code)
    assert r.similarity.is_monotonic_decreasing
    assert r.similarity.between(0,1.000001).all()
    assert r.occupation_code.is_unique
summary = pd.DataFrame([{'occupation_code':c, 'coverage':coverage.loc[c],
                         'best_similarity':recommend(c).iloc[0].similarity} for c in check_codes])
summary.to_csv(OUTPUT/'quality_checks.csv',index=False)
print('Passed invariants for',len(check_codes),'occupations')
print('Eligible occupation coverage:',f'{len(eligible)/len(X):.1%}')
print('Mean top-match similarity (tested subset):',round(summary.best_similarity.mean(),3))
print('These are NOT externally validated suitability or prediction accuracy metrics.')

## 7. Submission-ready notes
- **Runtime:** Python / Pandas / scikit-learn on an IBM LinuxONE-hosted Jupyter container.
- **Input:** O*NET occupational importance ratings; optional user self-reported confidence.
- **Output:** Top-five occupation similarity recommendations with observed shared/different requirements.
- **Privacy:** No names, contact details or personal data are required; notebook output can still retain typed skill ratings, so review before public sharing.
- **Limitations:** American occupational taxonomy, training/licensing requirements not verified for Australia, missing values imputed only for ranking, and no externally validated career-outcome labels.
- **Next:** Compare against an occupation-family baseline and add human review of results; create a short demo and README for submission.

**Suggested 20-second demo:** Select an occupation → load matches → adjust optional confidence sliders → show recommendations → explain similarity chart and skill-development flags.